# MMC v2026 numerical and native-adjoint audit

Development experiment for Ubuntu / RTX 4090. Keep `jacobian_quality_analysis.ipynb` for general archive QC.
This notebook computes only selected pairs, retaining raw configurations, logs, fields, histories, and derived sensitivities.
**No production Jacobian code is changed. No GPU results are claimed by this notebook's creation.**

Run setup and the small CUDA smoke stage first. Enable individual experiment switches afterward.
The default audit budget is **1e8 photons per run, three independent seeds**. A source/detector pair needs two fluence runs;
native adjoint runs launch both source and detector slots. Reusing a seed across releases/backends does not guarantee paired random streams.
Plots load matching caches without enabling simulation. Do not interpret a smoke run as an accuracy result.

## Run end to end on Ubuntu

1. **Select the environment.** Open this notebook in a Jupyter kernel using the installed `mmcnirs` checkout on
   `jacobian-calculation`, with NumPy/SciPy/Matplotlib. Have the NVIDIA driver and requested CUDA/OpenCL backends available.
   The resolver downloads the selected CLI runtimes when a run/preflight needs them. Native Python modes additionally
   require the matching v2026 `pmmc-trinity` wheel (`pmmc 0.6.0`, loadable `_rtmmc`) and `iso2mesh` installed in that kernel;
   CLI full FEM and cached analysis do not require that binding. No cell installs packages automatically.
2. **Set the inputs and budget in the next cell.** Point `MESH_PATH`, `PROBE_PATH`, and `OPTICAL_PATH` at existing prepared
   workflow outputs. Check `WAVELENGTH`, `PAIRS` (zero-based original probe indices), `GPU_ID`, `OUT`, `NPHOTON`, `SEEDS`,
   `SMOKE_PHOTONS`, and `MAX_DETECTED`. Defaults use one representative long channel, 1e8 photons and three seeds.
   `INCLUDE_SHORT=True` optionally adds one short channel. Output goes to ignored `notebooks/jacobian_audit/`.
3. **First run without transport.** Leave all `EXECUTE` entries and `REFRESH_HISTORY` false and run all cells top to bottom.
   Confirm input validation, tissue mapping and synthetic checks. Missing caches are expected on a first run.
4. **Run small smoke checks.** In the settings cell set `preflight`, `smoke`, `smoke_element`, and `smoke_native_cli` true;
   optionally enable `smoke_native_python` when its binding is installed. Rerun settings and cells through Stage 1.
   Inspect backend/version records, shapes, detector count/ESS, and native implementation residuals before continuing.
   Native residuals test implementation agreement, not measurement accuracy.
5. **Run the main matrix.** After successful smoke checks, set `versions`, `basis`, and `native_cli` true; optionally
   set `native_python` true. Rerun settings, then all cells in order. The version comparison uses `VERSION_BACKEND`
   for BOTH releases; the basis/native study uses `BACKEND`. Each pair/seed/formulation is cached. These are expensive
   runs; monitor the active notebook kernel. All runner calls use one attempt, with `TIMEOUT` as the limit.
6. **Review analysis.** Read `summary.csv`, `tissue_summary.csv`, the integrated comparisons and tissue fractions.
   Choose `TISSUE_ROW`, `BASE_ROW`, and `COMPARE_ROW` for the focused figures. Only raw results enter total-error
   comparisons; calibrated total agreement is a construction check. Native/manual emission geometries differ.
7. **Optional transport validation.** Set `PERTURB_TISSUE_ID` to a listed positive medium ID and
   `EXECUTE['perturbation']=True` after the CUDA nodal baseline exists. Check `PERTURB_PAIR` and `PERTURB_DELTAS`.
   Run the follow-up cell and review `transport_perturbation.csv`; beta remains fixed at baseline. This step is optional.
8. **Recover missing/truncated histories explicitly.** Keep all `EXECUTE` switches false, set `MAX_DETECTED` yourself
   to an adequate limit, and temporarily set `REFRESH_HISTORY=True`. Rerun settings and all cells in order through the desired smoke/matrix loader
   cell. Only cached standalone jobs with unavailable/invalid histories refresh their source simulation; detector fields
   and complete valid histories are reused. Refreshed source outputs/logs live in a new subdirectory, preserving originals.
   This flag applies to each affected standalone job visited by those cells. Set it false afterward. No limit is increased
   automatically and a failed refresh is not retried within the call. Perturbation histories are not refreshed by this flag.
9. **Replot later without MMC.** Return every `EXECUTE` switch and `REFRESH_HISTORY` to false and run top to bottom.
   Changes to `FD_STEPS`, calibration/analysis formulas, or plot selections recompute from cached transport. Field/spatial
   analysis remains available without histories; measurement, pathlength and calibration are explicitly unavailable.


In [ ]:
from pathlib import Path
import csv
import base64
import hashlib
import json
import platform
import sys
import zlib
import gzip
import importlib.metadata
import importlib
import subprocess
from scipy.special import logsumexp
import numpy as np
import matplotlib.pyplot as plt

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from mmcnirs.light_transport.prepare_jacobian_inputs import prepare_jacobian_inputs  # noqa: E402
from mmcnirs.mmc.runtime import get_mmc_executable  # noqa: E402
from mmcnirs.mmc.runner import run_mmc  # noqa: E402
from mmcnirs.mmc.history import read_flux, read_history  # noqa: E402
from mmcnirs.mmc.photons import compute_detected_photon_weights  # noqa: E402
from mmcnirs.mmc.jacobian import (  # noqa: E402
    _calculate_node_volumes,
    _compute_element_volumes,
    _interpolate_nodal_field_in_tetrahedron,
    _validate_complete_detected_history,
)
from mmcnirs.utils.jacobian_utils import build_jacobian_mmc_config, mmc_to_json  # noqa: E402

# Point these at prepared outputs from workflow_example/end_to_end_workflow.py.
DATA = ROOT / "legacy/pattern_cutting_data"
MESH_PATH = DATA / "mmcnirs_outputs/mesh.npz"
PROBE_PATH = DATA / "mmcnirs_outputs/probe.npz"
OPTICAL_PATH = DATA / "optical_properties.json"
WAVELENGTH = "850"
VERSION = "v2026"
BACKEND = "cuda"
# v2025.10 predates Trinity CUDA: compare releases on a common backend.
VERSION_BACKEND = "opencl"  # Both versions; do not compare old OpenCL vs new CUDA as a version effect.
GPU_ID = 1
NPHOTON = 100_000_000
SMOKE_PHOTONS = 100_000
MAX_DETECTED = 2_000_000  # Bound history memory; invalid histories are reported; recovery is opt-in.
SEEDS = [164833, 271831, 314159]
PAIRS = None  # Or [(source_index, detector_index), ...], ZERO-based original probe IDs.
INCLUDE_SHORT = False
RADIUS_MM = 1.0  # Matches production capture radius.
TEND_SECONDS = 5e-9  # One integrated fluence gate; increase to check truncation.
FD_STEPS = [1e-5, 3e-5, 1e-4]  # mm^-1; trajectory reweighting, no new simulations.
OUT = ROOT / "notebooks/jacobian_audit"
OUT.mkdir(parents=True, exist_ok=True)
EXECUTE = dict(smoke=False, versions=False, basis=False, native_cli=False, native_python=False)
# False means load matching cache only. Missing results remain visibly missing.
# native_python needs the v2026 pmmc-trinity wheel installed in THIS kernel (pmmc 0.6.0).
TIMEOUT = 1800
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print(platform.platform(), OUT)
# Additional stages are opt-in; no expansion of the expensive matrix.
EXECUTE.update(
    preflight=False, smoke_element=False, smoke_native_cli=False, smoke_native_python=False, perturbation=False
)
PERTURB_TISSUE_ID = None  # Set an actual positive MMC medium ID from the mapping below.
PERTURB_DELTAS = [1e-5, 3e-5, 1e-4]  # positive absorption changes, mm^-1
PERTURB_PAIR = 0  # index into PAIRS; one channel only
REFRESH_HISTORY = False  # Explicit source-only recovery for cached standalone fluence jobs.

## Conventions established from source

Reviewed tag **v2026 / 8bf9ef5e134a04ddf176f42161f648b3614f6fa0**:
[mesh normalization and output](https://github.com/fangq/mmc/blob/v2026/src/mmc_mesh.c),
[CUDA adjoint kernels](https://github.com/fangq/mmc/blob/v2026/src/mmc_core.cu),
[JSON parser](https://github.com/fangq/mmc/blob/v2026/src/mmc_utils.c),
[Python example](https://github.com/fangq/mmc/blob/v2026/pmmc/example/demo_mmclab_mesh_adjoint.py),
[MATLAB example](https://github.com/fangq/mmc/blob/v2026/mmclab/example/demo_mmclab_mesh_adjoint.m),
[release documentation](https://github.com/fangq/mmc/releases/tag/v2026).

* Coordinates are mm, absorption is mm⁻¹, `LengthUnit=1`; normalized, single-gate **fluence** is per launched energy, mm⁻².
  Do not multiply integrated fluence by the gate duration (flux would require that conversion).
* `mesh_normalize` corrects element fluence by volume, and weights deposited energy by medium absorption.
  The old extra absorption division caused medium-dependent scale errors. Nodal output uses tissue-only lumped volumes
  and an energy-balance normalizer. With `isnormalized=1`, exposed tissue nodes additionally receive
  the factor `2/(1+Reff)` (approximate planar-boundary correction; corners/curvature remain a limitation). Production Python instead integrates with all-tetrahedron V/4 weights.
* Native `jmua` is a **negative absolute adjoint bilinear sensitivity**, already volume integrated (mm⁻¹ for unit source).
  The kernel has no division by `Green_sd` or detected measurement. Dividing by an appropriate mm⁻² forward fluence yields
  a candidate log sensitivity in mm. Dividing it directly by dimensionless captured energy is dimensionally inconsistent.
* Full FEM computes each element's `-V/20 * ((sum s)(sum d) + dot(s,d))`, then distributes one quarter to each vertex.
  This preserves the integrated sensitivity, but is **not** an exact derivative with respect to each linearly interpolated
  nodal absorption parameter (which would involve triple basis products). Nodal approximation uses `-nvol*s*d`.
* JSON supports `Detector.Dir`, one-based `Optode.SDMap`, and `OutputType='adjoint'`.
  **The v2026 CLI does not parse `adjointmode`.** CLI runs below are default full FEM only.
  Both native modes are exercised through `pmmc.mmclab`, whose binding exposes `adjointmode`.
* `SDMap` limits output channels, not necessarily all source launches. Each experiment subsets to one pair so slot ordering
  is unambiguous: source then detector; Jacobian column 0 is original pair recorded in metadata.
* Native code gives the detector source its radius and direction. CUDA explicitly samples a uniform disk with a fixed inward direction. Whether this response matches the
  exiting-photon capture operator must be tested, not assumed. A pencil source at the detector center in production is a
  different measurement model. No fitted global scale is used to claim accuracy.

In [ ]:
def load_npz(path):
    with np.load(path, allow_pickle=False) as z:
        return {k: z[k] for k in z.files}


mesh, probe = load_npz(MESH_PATH), load_npz(PROBE_PATH)
optical = json.loads(OPTICAL_PATH.read_text())
inputs = prepare_jacobian_inputs(mesh, probe, optical, {"nphoton": NPHOTON}, WAVELENGTH)
nodes, elem, types = inputs.nodes, inputs.elements, inputs.element_tissue_ids
vol = _compute_element_volumes(nodes, elem)
tissue = types > 0
v_all = _calculate_node_volumes(nodes, elem)
v_tissue = np.zeros(len(nodes))
for k in range(4):
    np.add.at(v_tissue, elem[tissue, k], vol[tissue] / 4)
print("Non-tissue tets:", (~tissue).sum(), "nodes with different quadrature:", np.count_nonzero(v_all != v_tissue))
print("Volumes all/tissue (mm^3):", vol.sum(), vol[tissue].sum())

available_pairs = np.unique(np.asarray(probe["channel_pairings"], int), axis=0)
distances = np.linalg.norm(
    inputs.source_positions[available_pairs[:, 0]] - inputs.detector_positions[available_pairs[:, 1]], axis=1
)
if PAIRS is None:
    long = np.flatnonzero(distances >= 20)
    if not len(long):
        raise ValueError("No >=20 mm channel; configure PAIRS explicitly")
    chosen = [long[np.argmin(abs(distances[long] - np.median(distances[long])))]]
    short = np.flatnonzero(distances < 20)
    if INCLUDE_SHORT and len(short):
        chosen.append(short[np.argmin(abs(distances[short] - np.median(distances[short])))])
    PAIRS = [tuple(map(int, available_pairs[i])) for i in chosen]
for s, d in PAIRS:
    assert any(np.array_equal([s, d], p) for p in available_pairs)
    print(
        "Original pair",
        (s, d),
        "separation mm:",
        np.linalg.norm(inputs.source_positions[s] - inputs.detector_positions[d]),
    )
# Cache identity includes actual inputs, avoiding stale reuse after geometry/property edits.
input_id = hashlib.sha256()
for path in (MESH_PATH, PROBE_PATH, OPTICAL_PATH):
    input_id.update(path.read_bytes())
INPUT_ID = input_id.hexdigest()


def scatter_element(values):
    result = np.zeros(len(nodes))
    for k in range(4):
        np.add.at(result, elem[:, k], values / 4)
    return result


def fem_product(s, d, tissue_only=False):
    a, b = s[elem], d[elem]
    integral = vol / 20 * (a.sum(axis=1) * b.sum(axis=1) + (a * b).sum(axis=1))
    if tissue_only:
        integral = np.where(tissue, integral, 0)
    return scatter_element(-integral)


def at_detector(field, pair, basis):
    d = pair[1]
    e = int(inputs.detector_elements[d])
    if basis == 0:
        return float(field[e])
    return _interpolate_nodal_field_in_tetrahedron(inputs.detector_positions[d], e, nodes, elem, field)


# Reproduce mesh_getreff and the isnormalized=1 boundary nvol correction.
# This is an audit diagnostic, NOT a replacement for production quadrature.
def upstream_reff(n_in, n_out):
    theta = np.arange(1000) * (np.pi / 2000)
    co, si = np.cos(theta), np.sin(theta)
    rf = np.ones(1000)
    transmitted = theta < np.arcsin(1.0 / n_in)
    cp = np.sqrt(np.maximum(0, 1 - (n_in * si[transmitted]) ** 2))
    c = co[transmitted]
    rf[transmitted] = 0.5 * ((n_in * cp - n_out * c) / (n_in * cp + n_out * c)) ** 2
    rf[transmitted] += 0.5 * ((n_in * c - n_out * cp) / (n_in * c + n_out * cp)) ** 2
    rphi = np.sum(2 * si * co * rf) * np.pi / 2000
    rj = np.sum(3 * si * co * co * rf) * np.pi / 2000
    return (rphi + rj) / (2 - rphi + rj)


faces = np.concatenate([elem[tissue][:, face] for face in ([0, 1, 2], [0, 1, 3], [0, 2, 3], [1, 2, 3])])
owners = np.tile(np.flatnonzero(tissue), 4)
_, first, counts = np.unique(np.sort(faces, axis=1), axis=0, return_index=True, return_counts=True)
assert counts.max() <= 2, "Non-manifold tissue mesh"
boundary = first[counts == 1]
first_owner = np.full(len(nodes), len(elem), dtype=int)
for k in range(3):
    np.minimum.at(first_owner, faces[boundary, k], owners[boundary])
boundary_nodes = first_owner < len(elem)
v_mmc = v_tissue.copy()
for label in np.unique(types[first_owner[boundary_nodes]]):
    ids = np.flatnonzero(boundary_nodes)
    ids = ids[types[first_owner[ids]] == label]
    reff = upstream_reff(inputs.selected_properties[label, 3], inputs.selected_properties[0, 3])
    v_mmc[ids] *= 2 / (1 + reff)
print(
    "Boundary-normalized nodes:", boundary_nodes.sum(), "nvol sum / geometric volume:", v_mmc.sum() / vol[tissue].sum()
)
# History columns follow optical-property row IDs 1..N, NEVER node tissue labels.
MEDIUM_IDS = np.arange(1, len(inputs.selected_properties), dtype=int)
TISSUE_NAMES = dict(zip(map(int, mesh["ordered_tissue_ids"]), map(str, mesh["ordered_tissues"])))
print("Photon-history columns:", [(i, int(k), TISSUE_NAMES[int(k)]) for i, k in enumerate(MEDIUM_IDS)])


def volume_partition(elements, volumes, labels, ids, node_count):
    partition = np.zeros((node_count, len(ids)))
    for j, label in enumerate(ids):
        mask = labels == label
        for corner in range(4):
            np.add.at(partition[:, j], elements[mask, corner], volumes[mask] / 4)
    return partition


V_BY_TISSUE = volume_partition(elem, vol, types, MEDIUM_IDS, len(nodes))
np.testing.assert_allclose(V_BY_TISSUE.sum(axis=1), v_tissue)


def fem_elements(gs, gd):
    a, d = gs[elem], gd[elem]
    return -vol / 20 * (a.sum(axis=1) * d.sum(axis=1) + (a * d).sum(axis=1))


def element_tissue_sum(values):
    return np.array([values[types == k].sum() for k in MEDIUM_IDS])


def at_source(field, pair, basis):
    s = pair[0]
    e = int(inputs.source_elements[s])
    if basis == 0:
        return float(field[e])
    return _interpolate_nodal_field_in_tetrahedron(inputs.source_positions[s], e, nodes, elem, field)


def positive_ratio(a, denominator):
    return (
        np.asarray(a) / denominator
        if np.isfinite(denominator) and denominator > 0
        else np.full_like(np.asarray(a, dtype=float), np.nan)
    )


def calibrate(j, reference):
    length = -np.sum(j)
    beta = (
        float(reference / length)
        if np.isfinite(reference) and reference > 0 and np.isfinite(length) and length > 0
        else np.nan
    )
    return beta, beta * j

In [ ]:
# Small analytic checks: constant fields must integrate to volume for both quadratures.
assert np.isclose(v_all.sum(), vol.sum())
assert np.isclose(v_tissue.sum(), vol[tissue].sum())
assert np.isclose(-fem_product(np.ones(len(nodes)), np.ones(len(nodes))).sum(), vol.sum())
assert np.isclose(scatter_element(vol).sum(), vol.sum())
print("Quadrature constant-field checks passed.")

## Cached transport and freshly derived analysis

Transport identity contains geometry, optical properties, wavelength, runtime/backend, photon budget, seed,
source/detector model and time window. `FD_STEPS`, calibration, quadrature formulas and plots are excluded.
Existing directories are found through `settings.json`, ignoring the old schema and FD-step fields. Raw `.dat`,
`.mch` and native `.jnii` files are preferred; old `result.npz` contributes **only** `gs`, `gd`, and `native_raw`.
New Python-binding runs save only these transport arrays in `transport.npz`. All derived quantities are recomputed.
Existing files are preserved. Partial standalone runs resume only missing outputs. A missing history is explicitly
unsupported for analysis rather than silently rerunning an otherwise valid field cache.

History columns map directly to positive MMC optical-property row IDs. Total and individual-tissue reweighting use
the same saved photons; these are same-history derivative checks, not independent transport validation.
Negative-absorption central-difference sides remain NaN. A separate opt-in transport perturbation follows below.


Fluence loading requires fields only, not a photon history. Missing, unreadable or truncated histories leave
measurement/pathlength/calibration unavailable while retaining valid spatial analysis. `REFRESH_HISTORY=False`
keeps those caches untouched. Explicit source-only recovery writes a new `history-refresh-NNNN/` directory
using the current recording limit, then re-reads both source outputs. Complete valid histories are never refreshed.
The detector field is reused, including when it comes from a legacy NPZ. Only successfully validated refreshes
are selected automatically on subsequent cache-only loads; unsuccessful attempts remain preserved for inspection.


In [ ]:
executables = {}


def config_for(pair, photons, seed, basis=1, detector_source=False):
    s, d = pair
    cfg = build_jacobian_mmc_config(nodes, elem, types, inputs.selected_properties, photons)
    cfg.update(
        seed=int(seed),
        basisorder=basis,
        tend=TEND_SECONDS,
        tstep=TEND_SECONDS,
        srcpos=inputs.source_positions[s].tolist(),
        srcdir=inputs.source_directions[s].tolist(),
        e0=int(inputs.source_elements[s]) + 1,
        detpos=[[*inputs.detector_positions[d], RADIUS_MM]],
    )
    if detector_source:
        cfg.update(
            srcpos=inputs.detector_positions[d].tolist(),
            srcdir=inputs.detector_directions[d].tolist(),
            e0=int(inputs.detector_elements[d]) + 1,
            detpos=None,
            issavedet=0,
        )
    return cfg


def cli(cfg, directory, name, version, backend, native=False):
    if version not in executables:
        executables[version] = get_mmc_executable(version=version)
    document = json.loads(mmc_to_json(cfg))
    document["Session"]["ID"] = name
    if native:
        document["Session"].update(OutputType="adjoint", OutputFormat="jnii")
        document["Optode"]["Detector"][0]["Dir"] = [*inputs.detector_directions[cfg["_pair"][1]], 0.0]
        document["Optode"]["SDMap"] = [[1, 1]]
    path = directory / (name + ".json")
    path.write_text(json.dumps(document))
    result = run_mmc(
        path,
        working_directory=directory,
        executable=executables[version],
        timeout=TIMEOUT,
        max_trials=1,
        extra_args=["-c", backend, "-G", str(GPU_ID), "-H", str(min(cfg["nphoton"], MAX_DETECTED))],
    )
    (directory / (name + ".stdout.txt")).write_text(result.stdout)
    (directory / (name + ".stderr.txt")).write_text(result.stderr)
    return directory / name


def weighted_history(hist, photons, properties):
    _validate_complete_detected_history(hist, 0)
    assert np.all(np.asarray(hist["detid"]) == 1), "Unexpected detector ordering"
    weights = compute_detected_photon_weights(hist, properties)
    path_mm = np.asarray(hist["ppath"], float) * float(hist["unitinmm"])
    assert path_mm.shape[1] == len(MEDIUM_IDS), "History/property medium count mismatch"
    assert np.isfinite(path_mm).all() and (path_mm >= 0).all()
    total = weights.sum()
    partial = weights @ path_mm / total if total > 0 else np.full(len(MEDIUM_IDS), np.nan)
    paths = path_mm.sum(axis=1)
    fd = np.full((len(FD_STEPS), len(MEDIUM_IDS) + 1), np.nan)
    if total > 0:
        valid = weights > 0
        lw = np.log(weights[valid])
        for j, delta in enumerate(FD_STEPS):
            assert np.isfinite(delta) and delta > 0
            for k in range(len(MEDIUM_IDS) + 1):
                L = paths[valid] if k == 0 else path_mm[valid, k - 1]
                mua = properties[1:, 0] if k == 0 else properties[k : k + 1, 0]
                if np.any(mua - delta < 0):
                    continue  # physically invalid negative-absorption side; visible NaN
                fd[j, k] = -(logsumexp(lw - delta * L) - logsumexp(lw + delta * L)) / (2 * delta)
    return dict(
        measurement=total / photons,
        pathlength=partial.sum(),
        tissue_pathlength=partial,
        medium_ids=MEDIUM_IDS.copy(),
        tissue_names=np.array([TISSUE_NAMES[int(k)] for k in MEDIUM_IDS]),
        detected_count=len(weights),
        ess=total**2 / np.dot(weights, weights) if total > 0 else 0.0,
        reweighted_fd=fd[:, 0],
        reweighted_fd_tissue=fd[:, 1:],
    )


def history_metrics(stub, photons, properties=None):
    return weighted_history(
        read_history(stub.with_suffix(".mch")),
        photons,
        inputs.selected_properties if properties is None else properties,
    )


def read_native_array(path, expected_shape):
    # Narrow reader for this tagged CLI's CW JNIfTI payload. Dimensions are checked before reshaping.
    payload = json.loads(path.read_text())["NIFTIData"]
    shape = tuple(payload["_ArraySize_"])
    assert shape == expected_shape, (path, shape, expected_shape)
    dtype = {"single": "<f4", "double": "<f8"}[payload["_ArrayType_"]]
    if "_ArrayZipData_" in payload:
        raw = base64.b64decode(payload["_ArrayZipData_"])
        raw = {"zlib": zlib.decompress, "gzip": gzip.decompress}[payload["_ArrayZipType_"]](raw)
    else:
        raw = base64.b64decode(payload["_ArrayData_"])
    flat = np.frombuffer(raw, dtype=dtype)
    assert flat.size == np.prod(shape)
    # mesh_saveweightdata is slot-major; mesh_savejacob is channel-major, node fastest.
    result = flat.reshape(shape, order="F").astype(float)
    assert np.isfinite(result).all()
    return result


# Only transport choices enter the cache key. Old settings.json files are matched by
# their transport fields; schema and fd_steps are ignored. No old files are deleted.
TRANSPORT_KEYS = (
    "kind",
    "version",
    "backend",
    "basis",
    "pair",
    "seed",
    "photons",
    "radius",
    "wavelength",
    "tend",
    "input_id",
    "gpu_id",
    "pmmc_version",
    "perturbation",
)


def transport_identity(meta):
    return {k: meta.get(k) for k in TRANSPORT_KEYS}


def locate_cache(meta):
    identity = transport_identity(meta)
    key = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()[:20]
    target = OUT / (meta["kind"] + "-" + key)
    candidates = [target]
    for settings in sorted(OUT.glob(meta["kind"] + "-*/settings.json")):
        old = json.loads(settings.read_text())
        if transport_identity(old) == identity and settings.parent != target:
            candidates.append(settings.parent)

    # Prefer complete raw files, then extracted transport, then legacy result raw arrays.
    def score(directory):
        required = (
            ["source.dat", "detector.dat"]
            if meta["kind"] == "fluence"
            else ["source.dat", "source.mch"]
            if meta["kind"] == "perturbation"
            else ["native.jnii", "native_jmua.jnii"]
            if meta["kind"] == "native-cli"
            else []
        )
        field_count = sum((directory / x).is_file() for x in required)
        present = field_count + int(meta["kind"] == "fluence" and (directory / "source.mch").is_file())
        tier = (
            4
            if required and field_count == len(required)
            else 3
            if (directory / "transport.npz").is_file()
            else 2
            if (directory / "result.npz").is_file()
            else 1
            if present
            else 0
        )
        return tier, present

    return max(candidates, key=score)


preflight_records = {}


def preflight(version, backend, python_native=False):
    key = (version, backend, python_native)
    if key in preflight_records:
        return preflight_records[key]
    if backend not in {"cuda", "opencl", "sse"}:
        raise ValueError("Specify cuda, opencl or sse; no implicit backend substitution")
    record = dict(requested_version=version, requested_backend=backend, platform=platform.platform())
    if python_native:
        import pmmc

        extension = importlib.import_module("_rtmmc")  # must load against NVIDIA driver
        if getattr(pmmc, "_rtmmc", None) is not extension or not callable(getattr(extension, "run", None)):
            raise RuntimeError("pmmc is not routed to the loaded CUDA _rtmmc extension")
        package = importlib.metadata.version("pmmc")
        if package != "0.6.0" or version != "v2026" or backend != "cuda":
            raise RuntimeError("This native-binding audit requires v2026 pmmc 0.6.0 and explicit CUDA")
        record.update(package_version=package, module_file=pmmc.__file__, extension_file=extension.__file__)
        for name in ("version", "__version__", "build", "build_info"):
            value = getattr(extension, name, None)
            if value is not None:
                try:
                    record[name] = str(value() if callable(value) else value)
                except TypeError:
                    record[name] = str(value)
        # Actual backend usability is confirmed by the small native run below.
    else:
        executable = get_mmc_executable(version=version)
        executables[version] = executable
        record["executable"] = str(executable)
        # Explicit CLI metadata probes only; analysis remains in this kernel.
        proc = subprocess.run([str(executable), "--version"], capture_output=True, text=True, timeout=30)
        banner = proc.stdout + proc.stderr
        if version not in banner:
            raise RuntimeError(f"Runtime version mismatch: {banner}")
        record.update(version_output=banner, version_exit_code=proc.returncode)
        if backend != "sse":
            probe = subprocess.run([str(executable), "-c", backend, "-L"], capture_output=True, text=True, timeout=30)
            record.update(device_output=probe.stdout + probe.stderr, device_exit_code=probe.returncode)
            if probe.returncode != 0:
                raise RuntimeError(f"Requested backend/device probe failed: {record}")
    preflight_records[key] = record
    print("Preflight:", record)
    return record


def derive(raw, meta):
    # Called every time, even on a cache hit. Never read legacy J_* or old metrics.
    r = {k: np.asarray(v) for k, v in raw.items() if k in ("gs", "gd", "native_raw")}
    gs, gd = r["gs"], r["gd"]
    expected = len(nodes) if meta["basis"] else len(elem)
    assert gs.shape == gd.shape == (expected,) and np.isfinite(gs).all() and np.isfinite(gd).all()
    pair = meta["pair"]
    basis = meta["basis"]
    sd = at_detector(gs, pair, basis)
    ds = at_source(gd, pair, basis)
    sym = 0.5 * (sd + ds)
    r.update(green_sd=sd, green_ds=ds, green_sym=sym, reciprocity_discrepancy=float(positive_ratio(sd - ds, sym)))
    if min(sd, ds) <= 0 or not np.isfinite([sd, ds]).all():
        print("Invalid/zero point denominator:", meta["kind"], pair, sd, ds, "(no point relocation)")
    forms = {}

    def add(name, q, qt=None, status="raw", note=""):
        for denom_name, g in [("oneway", sd)] + ([("symmetric", sym)] if meta["kind"] == "fluence" else []):
            forms[name + "_" + denom_name] = dict(
                J=positive_ratio(q, g),
                tissue_L=None if qt is None else -positive_ratio(qt, g),
                status=status,
                note=note,
                numerator=name,
                denominator=denom_name,
            )

    if basis:
        prod = gs * gd
        add("J_lumped_all", -v_all * prod, status="diagnostic", note="includes non-tissue support")
        add("J_lumped_tissue", -v_tissue * prod, -(V_BY_TISSUE.T @ prod))
        add(
            "J_lumped_mmc_boundary",
            -v_mmc * prod,
            status="diagnostic",
            note="normalization weights; tissue allocation unsupported",
        )
        qe = fem_elements(gs, gd)
        add("J_fem_all", scatter_element(qe), status="diagnostic", note="all-element equal-quarter scatter")
        add(
            "J_fem_tissue",
            scatter_element(np.where(tissue, qe, 0)),
            element_tissue_sum(qe),
            note="same-field element reconstruction before visualization scatter",
        )
    else:
        qe = -vol * gs * gd
        add("J_element", scatter_element(qe), status="diagnostic", note="all-element support")
        add("J_element_tissue", scatter_element(np.where(tissue, qe, 0)), element_tissue_sum(qe))
    if "native_raw" in r:
        native = r["native_raw"]
        expected = -v_mmc * gs * gd if meta["kind"] == "python-1" else fem_product(gs, gd)
        residual = np.linalg.norm(native - expected) / max(np.linalg.norm(expected), 1e-30)
        r["native_formula_relative_error"] = residual
        assert native.shape == (len(nodes),) and np.isfinite(native).all() and native.max() <= 1e-12
        assert residual < 1e-3, f"Native implementation-consistency failure: {residual}"
        add(
            "J_native_point_denominator",
            native,
            status="diagnostic",
            note="native raw already volume integrated; tissue allocation unsupported directly",
        )
        # Native full FEM has a justified tissue reconstruction from its OWN fields.
        # Mode 1 uses corrected nvol; do not give it an invented physical tissue decomposition.
        if meta["kind"] != "python-1":
            forms["J_fem_tissue_oneway"]["note"] = "native full-FEM tissue reconstruction verified against native_raw"
    r["formulations"] = forms
    return r


def read_history_if_valid(stub, photons):
    try:
        return history_metrics(stub, photons), None
    except (OSError, ValueError, RuntimeError, AssertionError) as error:
        return {}, f"{type(error).__name__}: {error}"


def run_job(kind, version, backend, basis, pair, seed, photons, execute, *, refresh_history=None):
    meta = dict(
        kind=kind,
        version=version,
        backend=backend,
        basis=basis,
        pair=list(pair),
        seed=seed,
        photons=photons,
        radius=RADIUS_MM,
        wavelength=WAVELENGTH,
        tend=TEND_SECONDS,
        max_detected=MAX_DETECTED,
        input_id=INPUT_ID,
        gpu_id=GPU_ID,
        pmmc_version="0.6.0" if kind.startswith("python") else None,
    )
    directory = locate_cache(meta)
    if refresh_history is None:
        refresh_history = REFRESH_HISTORY
    # Completed source refreshes supersede only the source output, never the detector field.
    source_directory = directory
    if kind == "fluence":
        completed = sorted(directory.glob("history-refresh-*/complete.json"))
        if completed:
            source_directory = completed[-1].parent
    cfg = config_for(pair, photons, seed, basis)
    needed = (
        ["source.dat", "detector.dat"]
        if kind == "fluence"
        else ["native.jnii", "native_jmua.jnii"]
        if kind == "native-cli"
        else []
    )
    raw = None
    files_present = needed and all(
        ((source_directory if kind == "fluence" and x == "source.dat" else directory) / x).is_file() for x in needed
    )
    if files_present:
        if kind == "fluence":
            raw = dict(gs=read_flux(source_directory / "source.dat"), gd=read_flux(directory / "detector.dat"))
        else:
            field = read_native_array(directory / "native.jnii", (len(nodes), 1, 2))
            raw = dict(
                gs=field[:, 0, 0],
                gd=field[:, 0, 1],
                native_raw=read_native_array(directory / "native_jmua.jnii", (len(nodes), 1))[:, 0],
            )
    else:
        for filename in ("transport.npz", "result.npz"):
            if (directory / filename).is_file():
                old = load_npz(directory / filename)
                required = ("gs", "gd") + (() if kind == "fluence" else ("native_raw",))
                if all(k in old for k in required):
                    raw = {k: old[k] for k in required}
                    break
    if raw is not None and source_directory != directory:
        raw["gs"] = read_flux(source_directory / "source.dat")
    if raw is None:
        if not execute:
            print("Missing transport:", kind, version, backend, basis, pair, seed)
            return None, meta
        record = preflight(version, backend, kind.startswith("python"))
        directory.mkdir(parents=True, exist_ok=True)
        (directory / "settings.json").write_text(json.dumps(meta, indent=2))
        (directory / "preflight.json").write_text(json.dumps(record, indent=2))
        if kind == "fluence":
            if not (directory / "source.dat").is_file():
                cli(cfg, directory, "source", version, backend)
            if not (directory / "detector.dat").is_file():
                cli(config_for(pair, photons, seed + 7919, basis, True), directory, "detector", version, backend)
        elif kind == "native-cli":
            cfg["_pair"] = pair
            cli(cfg, directory, "native", version, backend, native=True)
        else:
            import pmmc

            cfg.update(
                outputtype="adjoint",
                adjointmode=int(kind[-1]),
                compute=backend,
                gpuid=GPU_ID,
                maxdetphoton=MAX_DETECTED,
                detdir=[[*inputs.detector_directions[pair[1]], 0.0]],
                sdmap=[[1, 1]],
            )
            (directory / "binding_config.json").write_text(json.dumps(cfg))
            native = pmmc.mmclab(cfg)
            field = np.asarray(native["flux"])
            jac = np.asarray(native["jmua"])
            assert field.shape == (len(nodes), 1, 2) and jac.shape == (len(nodes), 1)
            np.savez_compressed(directory / "transport.npz", gs=field[:, 0, 0], gd=field[:, 0, 1], native_raw=jac[:, 0])
        # Load the newly generated raw outputs through the identical cache path.
        return run_job(kind, version, backend, basis, pair, seed, photons, False, refresh_history=False)
    metrics, history_problem = {}, None
    if kind == "fluence":
        metrics, history_problem = read_history_if_valid(source_directory / "source", photons)
        if history_problem is not None and refresh_history:
            record = preflight(version, backend)
            # Preserve all original files and failed attempts. No expensive retries.
            attempts = list(directory.glob("history-refresh-*"))
            number = max([int(d.name.rsplit("-", 1)[1]) for d in attempts] + [0]) + 1
            refreshed = directory / f"history-refresh-{number:04d}"
            refreshed.mkdir(parents=True)
            (refreshed / "settings.json").write_text(json.dumps(meta, indent=2))
            (refreshed / "preflight.json").write_text(json.dumps(record, indent=2))
            cli(cfg, refreshed, "source", version, backend)
            # Always pair regenerated source fluence with its regenerated history.
            raw["gs"] = read_flux(refreshed / "source.dat")
            assert raw["gs"].shape == (len(nodes) if basis else len(elem),) and np.isfinite(raw["gs"]).all()
            source_directory = refreshed
            metrics, history_problem = read_history_if_valid(refreshed / "source", photons)
            if history_problem is None:
                (refreshed / "complete.json").write_text(json.dumps(dict(max_detected=MAX_DETECTED)))
        if history_problem is not None:
            print(
                "History unavailable; field analysis retained:",
                source_directory,
                history_problem,
                "Set REFRESH_HISTORY=True explicitly to rerun only this source with MAX_DETECTED.",
            )
    r = derive(raw, meta)
    r.update(metrics)
    r["history_status"] = (history_problem or "complete") if kind == "fluence" else "separate standalone reference"
    r["cache_directory"] = str(directory)
    r["source_directory"] = str(source_directory)
    return r, meta

## Stage 1 — preflight and small smoke checks

All switches default to false. `preflight` checks the resolved CLI version and requested GPU backend with `-L`;
actual CUDA operation is confirmed by smoke output, not by a banner alone. Expensive cache misses also invoke preflight.
`smoke`, `smoke_element`, and `smoke_native_cli` test separate small runs. `smoke_native_python` tests both modes only
when enabled and requires an importable CUDA `_rtmmc` actually used by `pmmc`. Module paths and exposed build/version
information are recorded. Missing optional bindings never block cached analysis or CLI runs. No packages are installed
and no backend is substituted. Review smoke output before enabling the expensive matrix.


In [ ]:
if EXECUTE["preflight"]:
    preflight(VERSION, BACKEND)
    if EXECUTE["native_python"] or EXECUTE["smoke_native_python"]:
        preflight(VERSION, BACKEND, True)
smoke_plans = [("fluence", 1, "smoke"), ("fluence", 0, "smoke_element"), ("native-cli", 1, "smoke_native_cli")]
if EXECUTE["smoke_native_python"]:
    smoke_plans += [("python-0", 1, "smoke_native_python"), ("python-1", 1, "smoke_native_python")]
for kind, basis, switch in smoke_plans:
    smoke, smoke_meta = run_job(kind, VERSION, BACKEND, basis, PAIRS[0], SEEDS[0], SMOKE_PHOTONS, EXECUTE[switch])
    if smoke is not None:
        print(
            "Smoke:",
            kind,
            basis,
            {
                k: smoke[k]
                for k in ("green_sd", "green_ds", "detected_count", "ess", "native_formula_relative_error")
                if k in smoke
            },
        )

## Stages 2 and 3 — experiment matrix

Release comparison: **both versions, same OpenCL backend, basis 1**. CUDA basis study: v2026, basis 0 and 1.
Native CLI: v2026 CUDA, default full FEM. Native Python: both modes using a separately installed **v2026 pmmc-trinity** wheel;
the runtime resolver does not install Python bindings. Install the matching wheel from the
[official release](https://github.com/fangq/mmc/releases/tag/v2026) in the workstation notebook environment. `pmmc.mmclab` also requires `iso2mesh`.
If unavailable, leave `native_python=False`: the CLI full-FEM experiment and local nodal approximation remain usable,
but the local approximation must not be reported as a native mode-1 run.

The matrix is intentionally repeated per selected pair. Raw single-source histories provide the reference for native runs
at the same pair/backend/seed. Native and separate runs have different Monte Carlo streams and detector source models.
Native slots share the total photon budget (random slot selection), so each field has roughly half
the photons of a separate fluence run. Compare repeat variability rather than assuming equal precision.

In [ ]:
results = []
for pair in PAIRS:
    for seed in SEEDS:
        plans = [("fluence", v, VERSION_BACKEND, 1, "versions") for v in ("v2025.10", VERSION)]
        plans += [("fluence", VERSION, BACKEND, b, "basis") for b in (1, 0)]
        plans += [("native-cli", VERSION, BACKEND, 1, "native_cli")]
        plans += [(f"python-{mode}", VERSION, BACKEND, 1, "native_python") for mode in (0, 1)]
        for kind, version, backend, basis, switch in plans:
            r, meta = run_job(kind, version, backend, basis, pair, seed, NPHOTON, EXECUTE[switch])
            if r is not None:
                results.append((meta, r))
print("Loaded/computed runs:", len(results))

## Stage 4 — total sensitivity versus tissue allocation

**Cedalion-style denominator on MMC fields:** `g_sd` and `g_ds` are evaluated at the original registered positions,
using barycentric interpolation (nodal) or containing elements (element fields). The one-way baseline and symmetric
alternative share exactly the same fields and quadrature. Zero/invalid denominators produce explicit NaNs, never relocation.
Native pencil/disk runs do not provide a pure reciprocity-error test; their reverse values are diagnostic only.

**Sassaroli-style global channel calibration:** `beta=L_det_total/(-sum(J))`. Only physical geometric, tissue-domain
formulations are calibrated. All modeled positive-ID tissues enter the sum; no cortex-only normalization or per-tissue
fitting occurs. One beta is fitted for each pair/wavelength/configuration/repeat, then held fixed. Total agreement is a
construction check. Positive scalar denominators cancel under calibration; the notebook asserts equality and stores only
one calibrated map per numerator. Tissue fractions must be unchanged. No extra area/index/boundary factor is fitted.

Tissue allocation uses element labels, never node labels: lumped products use `V[i,k]`, and FEM/element products retain
per-element contributions before scatter. Native full-FEM tissue allocation is explicitly a same-field reconstruction,
verified against native raw output. Native raw nodal and corrected-nvol outputs have **unsupported** physical tissue
allocation; their residual checks only reproduce implementation. Equal-quarter scatter is a visualization representation,
not an exact derivative for a different nodal absorption parameterization. Native/manual differences also include disk
versus pencil emission. Local formulas on the native fields isolate assembly without new transport runs.


In [ ]:
rows, maps, tissue_rows, calibrated_maps = [], [], [], {}


def calibrated_key(meta):
    return (meta["kind"], meta["version"], meta["backend"], meta["basis"], tuple(meta["pair"]), meta["seed"])


def summarize_run(meta, r, reference):
    ref = float(reference.get("pathlength", np.nan))
    ld = np.asarray(reference.get("tissue_pathlength", np.full(len(MEDIUM_IDS), np.nan)))
    out = []
    allocation = []
    calibrated = {}
    for name, form in r["formulations"].items():
        j = form["J"]
        length = -float(j.sum())
        physical = form["tissue_L"] is not None
        beta, jcal = calibrate(j, ref) if physical else (np.nan, np.full_like(j, np.nan))
        err = (length - ref) / ref if np.isfinite(ref) and ref > 0 else np.nan
        row = dict(
            version=meta["version"],
            backend=meta["backend"],
            kind=meta["kind"],
            basis=meta["basis"],
            formulation=name,
            pair=tuple(meta["pair"]),
            seed=meta["seed"],
            status=form["status"],
            note=form["note"],
            separation_mm=float(
                np.linalg.norm(inputs.source_positions[meta["pair"][0]] - inputs.detector_positions[meta["pair"][1]])
            ),
            integrated_mm=length,
            reference_mm=ref,
            signed_relative_error=err,
            absolute_relative_error=abs(err),
            raw_ratio=length / ref if ref > 0 else np.nan,
            beta=beta,
            green_sd=r["green_sd"],
            green_ds=r["green_ds"],
            green_sym=r["green_sym"],
            reciprocity_discrepancy=r["reciprocity_discrepancy"],
            reciprocity_interpretation="numerical + sampling"
            if meta["kind"] == "fluence"
            else "mixed pencil/disk profiles",
            native_implementation_residual=float(r.get("native_formula_relative_error", np.nan)),
            calibrated_total_construction_check=-float(jcal.sum()),
            measurement=float(reference.get("measurement", np.nan)),
            count=float(reference.get("detected_count", np.nan)),
            ess=float(reference.get("ess", np.nan)),
            history_status=reference.get("history_status", "unavailable"),
            tissue_status="supported" if physical else "unsupported",
        )
        out.append((row, j))
        if not physical:
            continue
        lj = form["tissue_L"]
        if np.isfinite(j).all():
            np.testing.assert_allclose(lj.sum(), length, rtol=1e-10, atol=1e-10)
        fj = positive_ratio(lj, length)
        fd = positive_ratio(ld, ref)
        if np.isfinite(beta):
            np.testing.assert_allclose(positive_ratio(beta * lj, beta * length), fj, rtol=1e-12, atol=1e-12)
            np.testing.assert_allclose(-jcal.sum(), ref, rtol=1e-12)
            key = form["numerator"]
            if key in calibrated:
                # Fixed numerator + global calibration cancels either positive denominator.
                np.testing.assert_allclose(calibrated[key], jcal, rtol=1e-11, atol=1e-12)
            else:
                calibrated[key] = jcal
        for k, label in enumerate(MEDIUM_IDS):
            if not np.any(types == label):
                continue
            allocation.append(
                dict(
                    version=meta["version"],
                    backend=meta["backend"],
                    kind=meta["kind"],
                    basis=meta["basis"],
                    formulation=name,
                    pair=tuple(meta["pair"]),
                    seed=meta["seed"],
                    medium_id=int(label),
                    tissue=TISSUE_NAMES[int(label)],
                    raw_mm=lj[k],
                    calibrated_mm=beta * lj[k],
                    reference_mm=ld[k],
                    fraction_J=fj[k],
                    fraction_det=fd[k],
                    beta=beta,
                    raw_error_mm=lj[k] - ld[k],
                    calibrated_error_mm=beta * lj[k] - ld[k],
                    note=form["note"],
                )
            )
    return out, allocation, calibrated


for meta, r in results:
    reference = (
        r
        if meta["kind"] == "fluence"
        else next(
            (
                rr
                for mm, rr in results
                if mm["kind"] == "fluence"
                and mm["basis"] == 1
                and all(mm[k] == meta[k] for k in ("version", "backend", "pair", "seed", "photons"))
            ),
            {},
        )
    )
    out, allocation, calibrated = summarize_run(meta, r, reference)
    for row, j in out:
        rows.append(row)
        maps.append(j)
    tissue_rows.extend(allocation)
    calibrated_maps[calibrated_key(meta)] = calibrated


def write_table(name, records):
    if records:
        with (OUT / name).open("w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=list(records[0]))
            writer.writeheader()
            writer.writerows(records)


write_table("summary.csv", rows)
write_table("tissue_summary.csv", tissue_rows)
print("Raw/diagnostic totals only; calibrated totals are construction checks, never an accuracy ranking.")
for row in rows:
    print(
        row["kind"],
        row["version"],
        row["pair"],
        row["seed"],
        row["formulation"],
        row["status"],
        {
            k: row[k]
            for k in (
                "green_sd",
                "green_ds",
                "reciprocity_discrepancy",
                "raw_ratio",
                "beta",
                "native_implementation_residual",
                "tissue_status",
            )
        },
    )
for meta, r in results:
    if "reweighted_fd" in r:
        print(
            "Same-history reweighting:",
            meta["pair"],
            meta["seed"],
            "steps",
            FD_STEPS,
            "total",
            r["reweighted_fd"],
            "by medium ID",
            r["medium_ids"],
            r["reweighted_fd_tissue"],
        )
if not rows:
    print("No transport caches available. All GPU switches remain disabled.")

In [ ]:
if rows:
    fig, ax = plt.subplots(1, 3, figsize=(14, 4), layout="constrained")
    groups = sorted(set((r["version"], r["backend"], r["kind"], r["basis"], r["formulation"], r["pair"]) for r in rows))
    for group in groups:
        rr = [
            r for r in rows if (r["version"], r["backend"], r["kind"], r["basis"], r["formulation"], r["pair"]) == group
        ]
        values = np.array([r["signed_relative_error"] for r in rr])
        finite = np.isfinite(values)
        label = "/".join(map(str, group))
        ax[0].scatter([r["reference_mm"] for r in rr], [r["integrated_mm"] for r in rr], s=15)
        if finite.any():
            ax[1].errorbar(
                rr[0]["separation_mm"],
                values[finite].mean(),
                yerr=values[finite].std(ddof=1) if finite.sum() > 1 else 0,
                fmt="o",
                label=label,
            )
            ax[2].scatter([r["ess"] for r in rr], abs(values), s=15)
            print(
                label,
                "mean signed error:",
                values[finite].mean(),
                "repeat SD:",
                values[finite].std(ddof=1) if finite.sum() > 1 else np.nan,
                "valid repeats:",
                finite.sum(),
            )
    limits = ax[0].get_xlim()
    ax[0].plot(limits, limits, "k--", lw=1)
    ax[0].set(xlabel="History mean path (mm)", ylabel="Integrated sensitivity (mm)")
    ax[1].axhline(0, color="k", lw=1)
    ax[1].set(xlabel="Source–detector distance (mm)", ylabel="Signed relative error")
    ax[2].set(xlabel="Detected-photon ESS", ylabel="Absolute relative error", xscale="symlog")
    ax[1].legend(fontsize=5, loc="upper left", bbox_to_anchor=(0, -0.25))
    fig.savefig(OUT / "integrated_comparison.png", bbox_inches="tight")
    plt.show()

In [ ]:
# Same-pair, same-seed, common-backend release comparison of forward fields.
def available_ratio(new, old, key):
    numerator, denominator = new.get(key, np.nan), old.get(key, np.nan)
    if not np.isfinite([numerator, denominator]).all() or numerator <= 0 or denominator <= 0:
        return "unavailable (missing, non-finite or non-positive value)"
    return float(numerator / denominator)


for old_meta, old in results:
    if old_meta["kind"] != "fluence" or old_meta["version"] != "v2025.10":
        continue
    new = next(
        (
            rr
            for mm, rr in results
            if mm["kind"] == "fluence"
            and mm["version"] == VERSION
            and all(mm[k] == old_meta[k] for k in ("basis", "backend", "pair", "seed"))
        ),
        None,
    )
    if new is None:
        continue
    for key in ("gs", "gd"):
        a, b = old[key], new[key]
        scale = np.dot(a, b) / np.dot(a, a) if np.dot(a, a) > 0 else np.nan
        residual = np.linalg.norm(b - scale * a) / np.linalg.norm(b) if np.linalg.norm(b) > 0 else np.nan
        print(
            old_meta["pair"],
            old_meta["seed"],
            key,
            "amplitude new/old:",
            scale,
            "scale-adjusted residual:",
            residual,
            "correlation:",
            np.corrcoef(a, b)[0, 1] if np.std(a) > 0 and np.std(b) > 0 else "unavailable (constant field)",
        )
    print(
        "Green_sd ratio:",
        available_ratio(new, old, "green_sd"),
        "captured measurement ratio:",
        available_ratio(new, old, "measurement"),
    )

## Tissue validation (raw and globally calibrated)

Choose one baseline channel/configuration/seed. Fractions test allocation between tissues; calibration does not change
them. Neither tissue agreement nor global calibration establishes spatial accuracy within gray matter. Unsupported
formulations stay in the total table but do not acquire invented tissue metrics. `tissue_summary.csv` contains every run.


In [ ]:
TISSUE_ROW = 0  # choose from supported raw formulations below
supported = [r for r in rows if r["tissue_status"] == "supported" and r["formulation"].endswith("_oneway")]
if supported:
    choice = supported[TISSUE_ROW]
    selected = [
        r
        for r in tissue_rows
        if all(r[k] == choice[k] for k in ("version", "backend", "kind", "basis", "formulation", "pair", "seed"))
    ]
    print(
        "Tissue selection:",
        choice["version"],
        choice["backend"],
        choice["kind"],
        choice["pair"],
        choice["seed"],
        choice["formulation"],
    )
    for r in selected:
        print(
            r["medium_id"],
            r["tissue"],
            {k: r[k] for k in ("raw_mm", "calibrated_mm", "reference_mm", "fraction_J", "fraction_det", "beta")},
        )
    x = np.arange(len(selected))
    fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
    for offset, key, label in [
        (-0.25, "raw_mm", "Raw"),
        (0, "calibrated_mm", "Global calibration"),
        (0.25, "reference_mm", "History"),
    ]:
        axes[0].bar(x + offset, [r[key] for r in selected], 0.25, label=label)
    axes[1].bar(x - 0.15, [r["fraction_J"] for r in selected], 0.3, label="J (raw = calibrated fraction)")
    axes[1].bar(x + 0.15, [r["fraction_det"] for r in selected], 0.3, label="History")
    for ax in axes:
        ax.set_xticks(x, [r["tissue"] for r in selected], rotation=30, ha="right")
        ax.legend(fontsize=8)
    axes[0].set_ylabel("Tissue-integrated sensitivity / pathlength (mm)")
    axes[1].set_ylabel("Fraction of full modeled tissue total")
    fig.savefig(OUT / "tissue_allocation.png", bbox_inches="tight")
    plt.show()

## Opt-in independent transport follow-up — one tissue, one channel

Set a positive `PERTURB_TISSUE_ID`, then enable `EXECUTE['perturbation']` only when ready. This adds one forward run
per positive perturbation size and existing seed, at the same geometry/backend/photon budget. The capture measurement is
always `sum(w)/N` for the same detector. Optical-property changes are applied both to transport and history weight recovery.
Require finite nonnegative absorption. Use the baseline **lumped tissue one-way** prediction and its baseline beta; beta is
not refitted. Report measured log changes, raw/calibrated first-order predictions, and repeat SD (not a confidence interval).
Common seeds do not guarantee common trajectories after changing absorption. Check convergence with perturbation size and
Monte Carlo noise. This tissue perturbation cannot validate spatial accuracy within gray matter; no regional claim is made.


In [ ]:
perturbation_rows = []
if PERTURB_TISSUE_ID is not None:
    label = int(PERTURB_TISSUE_ID)
    if label not in MEDIUM_IDS or not np.any(types == label):
        raise ValueError("Choose a tissue ID present in the actual element labels")
    pair = PAIRS[PERTURB_PAIR]
    for seed in SEEDS:
        baseline = next(
            (
                (m, r)
                for m, r in results
                if m["kind"] == "fluence"
                and m["basis"] == 1
                and m["version"] == VERSION
                and m["backend"] == BACKEND
                and m["pair"] == list(pair)
                and m["seed"] == seed
            ),
            None,
        )
        if baseline is None:
            print("Missing baseline; run/load basis=1 first:", pair, seed)
            continue
        bm, br = baseline
        M0 = float(br.get("measurement", np.nan))
        L0 = float(br.get("pathlength", np.nan))
        form = br["formulations"]["J_lumped_tissue_oneway"]
        beta, _ = calibrate(form["J"], L0)
        if not (np.isfinite(M0) and M0 > 0 and np.isfinite(beta)):
            print("Invalid baseline detector support:", pair, seed)
            continue
        derivative = -form["tissue_L"][np.flatnonzero(MEDIUM_IDS == label)[0]]
        for delta in PERTURB_DELTAS:
            if not np.isfinite(delta) or delta <= 0:
                raise ValueError("Perturbation sizes must be finite and positive")
            properties = inputs.selected_properties.copy()
            properties[label, 0] += delta
            if not np.isfinite(properties).all() or np.any(properties[:, 0] < 0):
                raise ValueError("Nonphysical absorption")
            pm = dict(bm, kind="perturbation", perturbation={"medium_id": label, "delta_mua": float(delta)})
            directory = locate_cache(pm)
            if not (directory / "source.mch").is_file():
                if not EXECUTE["perturbation"]:
                    print("Missing perturbation cache:", label, delta, seed)
                    continue
                record = preflight(VERSION, BACKEND)
                directory.mkdir(parents=True, exist_ok=True)
                (directory / "settings.json").write_text(json.dumps(pm, indent=2))
                (directory / "preflight.json").write_text(json.dumps(record, indent=2))
                cfg = config_for(pair, NPHOTON, seed, 1)
                cfg["prop"] = properties.tolist()
                cli(cfg, directory, "source", VERSION, BACKEND)
            measured = history_metrics(directory / "source", NPHOTON, properties)
            M = measured["measurement"]
            change = np.log(M / M0) if M > 0 else np.nan
            perturbation_rows.append(
                dict(
                    pair=pair,
                    seed=seed,
                    tissue=TISSUE_NAMES[label],
                    delta=delta,
                    basis=bm["basis"],
                    baseline_beta=beta,
                    baseline_measurement=M0,
                    measurement=M,
                    log_change=change,
                    raw_prediction=derivative * delta,
                    calibrated_prediction=beta * derivative * delta,
                    raw_log_residual=change - derivative * delta,
                    calibrated_log_residual=change - beta * derivative * delta,
                    count=measured["detected_count"],
                    ess=measured["ess"],
                )
            )
write_table("transport_perturbation.csv", perturbation_rows)
for delta in PERTURB_DELTAS:
    selected = [r for r in perturbation_rows if r["delta"] == delta]
    if selected:
        for metric in (
            "log_change",
            "raw_prediction",
            "calibrated_prediction",
            "raw_log_residual",
            "calibrated_log_residual",
        ):
            values = np.array([r[metric] for r in selected])
            values = values[np.isfinite(values)]
            print(
                "Independent transport:",
                delta,
                metric,
                "valid repeats",
                len(values),
                "mean",
                values.mean() if len(values) else np.nan,
                "repeat SD",
                values.std(ddof=1) if len(values) > 1 else np.nan,
            )

## Spatial shape versus amplitude

Choose any two rows from the printed summary (same pair recommended). Element sensitivities are conservatively distributed
V/4 to nodes for this common-space visualization; that projection smooths element structure and is not a recovered nodal derivative.
Raw amplitudes share a color scale. The difference uses one global baseline scale rather than dividing by near-zero nodes.
Shape comparison uses unit-total sensitivity; amplitude and total ratios remain separate. Correlation alone is not accuracy.

In [ ]:
BASE_ROW = 0
COMPARE_ROW = 1
MAX_POINTS = 12000
valid_map_selection = len(rows) > max(BASE_ROW, COMPARE_ROW) and all(
    np.isfinite(maps[i]).all() and -maps[i].sum() > 0 for i in (BASE_ROW, COMPARE_ROW)
)
if rows and not valid_map_selection:
    print("Spatial comparison unsupported for selected rows: invalid/zero sensitivity or denominator.")
if valid_map_selection:
    a, b = -maps[BASE_ROW], -maps[COMPARE_ROW]
    assert rows[BASE_ROW]["pair"] == rows[COMPARE_ROW]["pair"], "Select the same channel"
    scale = np.dot(a, b) / np.dot(a, a) if np.dot(a, a) > 0 else np.nan
    corr = np.corrcoef(a, b)[0, 1] if np.std(a) > 0 and np.std(b) > 0 else np.nan
    ashape, bshape = a / a.sum(), b / b.sum()
    print(
        "Spatial correlation:",
        corr,
        "least-squares amplitude b/a:",
        scale,
        "total b/a:",
        b.sum() / a.sum(),
        "unit-total shape L1:",
        abs(ashape - bshape).sum(),
    )
    # Same deterministic subset and limits in every panel; coordinates are actual mesh nodes.
    ids = np.linspace(0, len(nodes) - 1, min(MAX_POINTS, len(nodes)), dtype=int)
    delta = (b - a) / max(a.max(), 1e-30)
    fig = plt.figure(figsize=(13, 4), layout="constrained")
    for i, (value, title) in enumerate(
        [(a, "Baseline magnitude"), (b, "Comparison magnitude"), (delta, "Difference / baseline peak")]
    ):
        ax = fig.add_subplot(1, 3, i + 1, projection="3d")
        opts = (
            dict(cmap="viridis", vmin=0, vmax=max(a.max(), b.max()))
            if i < 2
            else dict(cmap="coolwarm", vmin=-max(abs(delta).max(), 1e-30), vmax=max(abs(delta).max(), 1e-30))
        )
        p = ax.scatter(*nodes[ids].T, c=value[ids], s=2, **opts)
        ax.set(title=title, xlabel="x (mm)", ylabel="y (mm)", zlabel="z (mm)")
        ax.set_box_aspect(np.ptp(nodes, axis=0))
        fig.colorbar(p, ax=ax, shrink=0.6, label="mm" if i < 2 else "relative")
    fig.savefig(OUT / "spatial_comparison.png", bbox_inches="tight")
    plt.show()

## Small deterministic analysis checks

Synthetic arrays and temporary legacy cache files only. These do not invoke MMC or alter saved experiments.

In [ ]:
# Small deterministic checks; temporary fixtures only, no MMC calls or audit-cache writes.
import tempfile
import struct
from unittest.mock import patch

# Two tetrahedra share nodes across a tissue interface. Node-label binning would fail here.
te = np.array([[0, 1, 2, 3], [0, 1, 2, 4]])
tv = np.array([2.0, 6.0])
tl = np.array([1, 2])
pv = volume_partition(te, tv, tl, np.array([1, 2]), 5)
np.testing.assert_allclose(pv[0], [0.5, 1.5])
np.testing.assert_allclose(pv.sum(axis=0), tv)

q = -np.array([2.0, 3.0, 5.0])
target = 37.0
beta1, j1 = calibrate(q / 2.0, target)
beta2, j2 = calibrate(q / 9.0, target)
np.testing.assert_allclose(j1, j2, rtol=1e-14)
np.testing.assert_allclose(-j1.sum(), target)
np.testing.assert_allclose(j1 / j1.sum(), q / q.sum())
assert np.isnan(positive_ratio(q, 0)).all()

# Real history parser fixture with distinct partial paths in each medium column.
nmedia = len(MEDIUM_IDS)
paths = np.arange(1, 1 + 3 * nmedia, dtype=float).reshape(3, nmedia) * 0.2
hist = dict(
    detid=np.ones(3),
    ppath=paths,
    w0=np.ones(3),
    unitinmm=2.0,
    detected_counts=np.array([3]),
    saved_counts=np.array([3]),
)
props = inputs.selected_properties.copy()
weights = compute_detected_photon_weights(hist, props)
expected = weights @ (2 * paths) / weights.sum()
hm = weighted_history(hist, 1000, props)
np.testing.assert_allclose(hm["tissue_pathlength"], expected)
np.testing.assert_allclose(hm["pathlength"], expected.sum())
np.testing.assert_array_equal(hm["medium_ids"], MEDIUM_IDS)
assert list(hm["tissue_names"]) == [TISSUE_NAMES[int(k)] for k in MEDIUM_IDS]
try:
    weighted_history(dict(hist, saved_counts=np.array([2])), 1000, props)
    raise AssertionError("Incomplete history was accepted")
except RuntimeError:
    pass

# Native and local same-field reconstructions, including tissue sums before scatter.
gs = np.full(len(nodes), 0.01)
gd = np.full(len(nodes), 0.02)
m = dict(
    kind="fluence",
    version=VERSION,
    backend=BACKEND,
    basis=1,
    pair=list(PAIRS[0]),
    seed=SEEDS[0],
    photons=1000,
    radius=RADIUS_MM,
    wavelength=WAVELENGTH,
    tend=TEND_SECONDS,
    max_detected=MAX_DETECTED,
    input_id=INPUT_ID,
    gpu_id=GPU_ID,
    pmmc_version=None,
)
r = derive(dict(gs=gs, gd=gd), m)
out, allocation, cal = summarize_run(m, r, hm)
assert allocation and cal
assert all(
    np.isclose(
        x["raw_mm"] / sum(t["raw_mm"] for t in allocation if t["formulation"] == x["formulation"]), x["fraction_J"]
    )
    for x in allocation
)
native = derive(dict(gs=gs, gd=gd, native_raw=fem_product(gs, gd)), dict(m, kind="native-cli"))
assert native["native_formula_relative_error"] < 1e-12
nodal = derive(dict(gs=gs, gd=gd, native_raw=-v_mmc * gs * gd), dict(m, kind="python-1"))
assert nodal["formulations"]["J_native_point_denominator_oneway"]["tissue_L"] is None

# Cache recovery from the OLD directory identity; deliberately stale derived J values.
# A complete raw file set remains usable even without result.npz.
original_out, original_steps = OUT, FD_STEPS
try:
    with tempfile.TemporaryDirectory(prefix="mmc-audit-cache-check-") as tmp:
        OUT = Path(tmp)
        old = OUT / "fluence-old-schema"
        old.mkdir()
        (old / "settings.json").write_text(json.dumps(dict(m, schema=2, fd_steps=[0.99])))
        np.savetxt(old / "source.dat", np.column_stack((np.arange(1, len(nodes) + 1), gs)))
        np.savetxt(old / "detector.dat", np.column_stack((np.arange(1, len(nodes) + 1), gd)))
        records = np.column_stack((np.ones(3), paths, np.ones(3))).astype("<f4")
        header = struct.pack(
            "<4s7IfIfi4I", b"MCXH", 1, nmedia, 1, nmedia + 2, 1000, 3, 3, 2.0, 0, 1.0, 1, 1, 0x45, 0, 0
        )
        (old / "source.mch").write_bytes(header + records.tobytes())
        np.savez(old / "result.npz", gs=gs, gd=gd, J_lumped_tissue=np.full(len(nodes), 999), pathlength=-999)
        with patch.dict(
            globals(),
            cli=lambda *a, **k: (_ for _ in ()).throw(AssertionError("Transport called")),
            preflight=lambda *a, **k: (_ for _ in ()).throw(AssertionError("Preflight called")),
        ):
            cached, _ = run_job("fluence", VERSION, BACKEND, 1, PAIRS[0], SEEDS[0], 1000, True)
            np.testing.assert_allclose(cached["tissue_pathlength"], expected, rtol=1e-6)
            assert cached["formulations"]["J_lumped_tissue_oneway"]["J"].max() <= 0
            FD_STEPS = [1e-6, 2e-6]
            again, _ = run_job("fluence", VERSION, BACKEND, 1, PAIRS[0], SEEDS[0], 1000, True)
            assert again["reweighted_fd_tissue"].shape == (2, nmedia)
            (old / "source.dat").unlink()
            (old / "detector.dat").unlink()
            fallback, _ = run_job("fluence", VERSION, BACKEND, 1, PAIRS[0], SEEDS[0], 1000, False)
            np.testing.assert_allclose(
                fallback["formulations"]["J_lumped_tissue_oneway"]["J"],
                cached["formulations"]["J_lumped_tissue_oneway"]["J"],
            )
            finite = np.isfinite(again["reweighted_fd_tissue"][0])
            np.testing.assert_allclose(again["reweighted_fd_tissue"][0, finite], expected[finite], rtol=1e-6)
            (old / "result.npz").unlink()
            assert locate_cache(m) == old, "Partial old transport should remain discoverable"

finally:
    OUT, FD_STEPS = original_out, original_steps
print("Synthetic tissue, history, native reconstruction, calibration and legacy-cache checks passed.")


# Corrective cache/recovery cases: all simulations below are mocked.
def check_field_only_and_recovery():
    with tempfile.TemporaryDirectory(prefix="mmc-audit-recovery-check-") as tmp:
        root = Path(tmp)
        directory = root / "fluence-legacy"
        directory.mkdir()
        (directory / "settings.json").write_text(json.dumps(dict(m, schema=2, fd_steps=[0.5], max_detected=2)))
        source_values = np.full(len(nodes), 0.01)
        detector_values = np.full(len(nodes), 0.02)

        def field(path, values):
            np.savetxt(path, np.column_stack((np.arange(1, len(values) + 1), values)))

        def history(path, saved=3):
            records = np.column_stack((np.ones(3), paths, np.ones(3))).astype("<f4")
            header = struct.pack(
                "<4s7IfIfi4I", b"MCXH", 1, nmedia, 1, nmedia + 2, 1000, 3, saved, 2.0, 0, 1.0, 1, 1, 0x45, 0, 0
            )
            path.write_bytes(header + records[:saved].tobytes())

        field(directory / "source.dat", source_values)
        field(directory / "detector.dat", detector_values)

        def forbidden(*args, **kwargs):
            raise AssertionError("Unexpected transport/preflight")

        with patch.dict(globals(), OUT=root, REFRESH_HISTORY=False, MAX_DETECTED=2, cli=forbidden, preflight=forbidden):
            # Raw fields only: even execute=True must not regenerate transport for a missing history.
            fields, _ = run_job("fluence", VERSION, BACKEND, 1, PAIRS[0], SEEDS[0], 1000, True)
            assert "measurement" not in fields and "pathlength" not in fields
            assert "FileNotFoundError" in fields["history_status"]
            np.testing.assert_array_equal(fields["gs"], source_values)
            assert np.isfinite(fields["formulations"]["J_lumped_tissue_oneway"]["J"]).all()
            totals, _, cal = summarize_run(m, fields, fields)
            assert not cal and all(np.isnan(row["beta"]) for row, _ in totals)
            assert available_ratio(fields, fields, "measurement").startswith("unavailable")
            assert available_ratio({"green_sd": 1.0}, {"green_sd": 0.0}, "green_sd").startswith("unavailable")
            assert available_ratio({"green_sd": 1.0}, {"green_sd": np.nan}, "green_sd").startswith("unavailable")
            # Legacy NPZ: old measurement values must NOT masquerade as recovered history metrics.
            (directory / "source.dat").unlink()
            (directory / "detector.dat").unlink()
            np.savez(
                directory / "result.npz", gs=source_values, gd=detector_values, measurement=999.0, pathlength=999.0
            )
            legacy, _ = run_job("fluence", VERSION, BACKEND, 1, PAIRS[0], SEEDS[0], 1000, False)
            assert "measurement" not in legacy and "pathlength" not in legacy
            np.testing.assert_array_equal(legacy["gd"], detector_values)
            field(directory / "source.dat", source_values)
            field(directory / "detector.dat", detector_values)
            history(directory / "source.mch", saved=2)
            truncated, _ = run_job("fluence", VERSION, BACKEND, 1, PAIRS[0], SEEDS[0], 1000, True)
            assert "truncated" in truncated["history_status"] and "measurement" not in truncated

        original_source = (directory / "source.dat").read_bytes()
        original_history = (directory / "source.mch").read_bytes()
        original_detector = (directory / "detector.dat").read_bytes()
        calls = []

        def fake_source(cfg, dest, name, version, backend, native=False):
            assert name == "source" and not native
            calls.append((name, MAX_DETECTED, cfg["nphoton"]))
            field(dest / "source.dat", source_values * 1.5)
            history(dest / "source.mch")
            return dest / name

        with patch.dict(
            globals(),
            OUT=root,
            REFRESH_HISTORY=False,
            MAX_DETECTED=10,
            cli=fake_source,
            preflight=lambda *a, **k: dict(mock=True),
        ):
            refreshed, _ = run_job(
                "fluence", VERSION, BACKEND, 1, PAIRS[0], SEEDS[0], 1000, False, refresh_history=True
            )
            assert calls == [("source", MAX_DETECTED, 1000)]
            assert refreshed["history_status"] == "complete" and refreshed["detected_count"] == 3
            np.testing.assert_allclose(refreshed["gs"], source_values * 1.5)
            np.testing.assert_array_equal(refreshed["gd"], detector_values)
            np.testing.assert_allclose(refreshed["tissue_pathlength"], expected, rtol=1e-6)
            np.testing.assert_allclose(refreshed["green_sd"], 1.5 * fields["green_sd"])
            np.testing.assert_allclose(refreshed["green_sym"], 0.5 * (0.015 + 0.02))
            assert (directory / "source.dat").read_bytes() == original_source
            assert (directory / "source.mch").read_bytes() == original_history
            assert (directory / "detector.dat").read_bytes() == original_detector
            # Complete regenerated outputs are reusable even if refresh remains explicitly enabled.
            reused, _ = run_job("fluence", VERSION, BACKEND, 1, PAIRS[0], SEEDS[0], 1000, False, refresh_history=True)
            assert len(calls) == 1 and reused["history_status"] == "complete"
            # Nodal and element maps coexist for identical version/backend/channel/seed.
            element_meta = dict(m, basis=0)
            er = derive(dict(gs=np.full(len(elem), 0.01), gd=np.full(len(elem), 0.02)), element_meta)
            nrows, ntissues, ncal = summarize_run(m, refreshed, refreshed)
            erows, etissues, ecal = summarize_run(element_meta, er, refreshed)
            stored = {calibrated_key(m): ncal, calibrated_key(element_meta): ecal}
            assert len(stored) == 2 and all(stored.values())
            assert all(row["basis"] == 1 for row, _ in nrows) and all(row["basis"] == 0 for row, _ in erows)
            assert all(row["basis"] == 1 for row in ntissues) and all(row["basis"] == 0 for row in etissues)
    print("Four corrective cases passed: field-only, legacy fields, source-only history recovery, basis coexistence.")


check_field_only_and_recovery()

## Interpretation and workstation checklist

**Verified during development:** source-level normalization and formulas; serializer extension keys; one-based SDMap;
CLI limitation on mode selection; local quadrature checks and successful cache-only execution with prepared pattern-cutting data
(selected pair 5,10; 33.14 mm; no non-tissue tetrahedra in that mesh). The development host is macOS ARM64, with no CUDA.
**Pending:** real CUDA smoke, real native CLI/JNIfTI and binding outputs, and all 1e8-photon accuracy comparisons.
Strict output shape/formula guards deliberately stop if the workstation binding or CLI differs from the reviewed tag.

1. Enable smoke, inspect raw logs and detector support, then run the basis and common-backend version matrix.
2. Enable native CLI; verify the native formula residual. Install the matching binding and enable both Python modes.
3. Compare **same-backend release contrasts**, then basis/geometry contrasts at v2026; increase repeats/photon count where
   repeat spread is comparable to a claimed improvement. Same seed across two engines does not eliminate Monte Carlo noise.
4. Inspect `Green_sd` and `measurement` separately. Their ratio has area units and is not a universal normalization constant.
   Finite-aperture capture, source angular distribution and boundary embedding can explain a remaining mismatch.
   Sweep `RADIUS_MM` and `TEND_SECONDS` with the same pair: the cache identity separates these experiments.
5. Add a short pair using `INCLUDE_SHORT=True` or explicit `PAIRS`; a single distance cannot establish distance dependence.
6. Inspect tissue-only versus all-element totals. Full FEM's scatter and nodal lumping have different spatial semantics even
   when their sums agree. Do not modify production volumes or label native raw output as Rytov without measurement validation.
7. The reweighting reference is consistent with the recorded detector operator but shares those photons. It cannot validate
   angular reciprocity, missing boundary weights, or transport changes independently; a true replay/transport finite difference
   remains a follow-up if disagreement persists. Check time-window truncation and roulette effects before interpreting reweighting.

**Conclusion to fill after execution:** which formulation reduces absolute pathlength error beyond repeat variability,
whether the change is spatial or global, whether it persists across separation/aperture, and what measurement mismatch remains.
No accuracy winner is selected in advance.

### Added analysis gates
* Cached transport reuse ignores old analysis fields, while freshly computed native residuals remain mandatory.
* Medium-column mapping, tissue-volume partition, global calibration invariance and denominator cancellation have
  passing synthetic checks in the preceding cell. No current real GPU audit caches were available during development.
* Total calibration agreement is a construction identity, not an accuracy metric. Compare tissue errors/fractions and
  the opt-in transport perturbation with beta held fixed instead. One long and one short pair are an initial audit,
  not a distance-dependence study.
* Actual CUDA device probing, `_rtmmc` usability, real output decoding/formula residuals and independent perturbation
  statistics still require workstation execution. No new simulations are enabled by this extension.
